In [1]:
# Fully reinstalling the compatible packages
!pip install numpy==1.23.5 sympy==1.12 --force-reinstall --quiet
!pip install farm-haystack[colab] --quiet --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 23.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blosc2 3.2.1 requires numpy>=1.26, but you have numpy 1.23.5 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
xarray 2025.1.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
chex 0.1.89 requires numpy>=1.24.1, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.23 requires numpy>=1.24.4, but you have numpy 1.23.5 

In [1]:
from haystack.nodes import FARMReader
import haystack

print("Haystack version:", haystack.__version__)


Haystack version: 1.26.4.post0


In [2]:
# Changing logging level to INFO to monitor training clearly
import logging
logging.basicConfig(level=logging.INFO)

# This ensures Haystack prints clear logs during training and usage.

In [3]:
# 2. Understanding the Telemetry Feature

# Disabling telemetry to avoid sending usage stats (optional step)
from haystack.telemetry import telemetry

# Disabling telemetry tracking
telemetry.enabled = False

# Haystack includes telemetry to collect anonymous usage stats and improve its tools.
# We can disable it if we're training locally or want full privacy, like we’re doing here.

# This step is optional, but it’s good practice to know what data is being shared and how to turn it off.

In [4]:
# 3. Getting Familiar with the FARMReader Component

# Importing the FARMReader class from Haystack
from haystack.nodes import FARMReader

# Initializing a pre-trained QA model as the reader
reader = FARMReader(model_name_or_path="deepset/roberta-base-squad2", use_gpu=True)

# FARMReader is the component responsible for answering questions from a given context.
# Here we load a model that’s already trained on SQuAD2 — a popular QA dataset.
# We’ll fine-tune it later to make it smarter in a custom domain.

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [22]:
# Removing the broken file (optional but clean)

!rm -f data/small_squad_dataset.json

In [13]:
# 4. Preparing the Dataset

# Downloading a small SQuAD-style dataset provided by deepset
# !wget -O small_squad_dataset.json https://raw.githubusercontent.com/deepset-ai/haystack/main/tutorials/small_squad_dataset.json

# !wget -O data/small_squad_dataset.json https://raw.githubusercontent.com/deepset-ai/haystack/main/tutorials/small_squad_dataset.json

# !wget https://raw.githubusercontent.com/deepset-ai/haystack/main/tutorials/small_squad_dataset.json -P data/

# This file contains question-answer pairs in the SQuAD format: context, question, answer, and start index.

--2025-04-05 11:14:35--  https://raw.githubusercontent.com/deepset-ai/haystack/main/tutorials/small_squad_dataset.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-04-05 11:14:35 ERROR 404: Not Found.



In [14]:
# Double-checking the file is not empty

# !head data/small_squad_dataset.json

head: cannot open 'data/small_squad_dataset.json' for reading: No such file or directory


In [15]:
# Creating the folder coz it doesn’t exist

# import os
# os.makedirs("data", exist_ok=True)

In [16]:
#  Downloading the file with full path
# !curl -o data/small_squad_dataset.json https://raw.githubusercontent.com/deepset-ai/haystack/main/tutorials/small_squad_dataset.json

# curl is often more stable than wget in Colab, and this saves directly into the data/ folder.

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0     69      0 --:--:-- --:--:-- --:--:--    70


In [18]:
# verifying the content
# !head data/small_squad_dataset.json

404: Not Found

In [31]:
# # since previous files didn't work, I'll create the JSON file directly
# # a valid SQuAD-style dataset into the data/ folder:

# import os
# import json

# # Creating the directory if it doesn't exist
# os.makedirs("data", exist_ok=True)

# # Creating a tiny valid SQuAD-style dataset
# sample_data = {
#     "data": [
#         {
#             "title": "Haystack",
#             "paragraphs": [
#                 {
#                     "context": "Haystack is an open-source framework by deepset for building search systems.",
#                     "qas": [
#                         {
#                             "question": "Who created Haystack?",
#                             "id": "1",
#                             "answers": [
#                                 {
#                                     "text": "deepset",
#                                     "answer_start": 41
#                                 }
#                             ],
#                             "is_impossible": False
#                         }
#                     ]
#                 }
#             ]
#         }
#     ]
# }

# # Saving the JSON file
# with open("data/small_squad_dataset.json", "w") as f:
#     json.dump(sample_data, f)

In [36]:
# Overwriting the dataset with valid format
import os
import json

# Create folder
os.makedirs("data", exist_ok=True)

# Define working SQuAD v1-style dataset
data = {
    "version": "1.1",
    "data": [
        {
            "title": "Haystack",
            "paragraphs": [
                {
                    "context": "Haystack is an open-source framework created by deepset for building production-ready search systems powered by LLMs.",
                    "qas": [
                        {
                            "id": "1",
                            "question": "Who created Haystack?",
                            "answers": [
                                {
                                    "text": "deepset",
                                    "answer_start": 43
                                }
                            ],
                            "is_impossible": False
                        }
                    ]
                }
            ]
        }
    ]
}

# Save dataset
with open("data/small_squad_dataset.json", "w") as f:
    json.dump(data, f)

In [37]:
!head data/small_squad_dataset.json

{"version": "1.1", "data": [{"title": "Haystack", "paragraphs": [{"context": "Haystack is an open-source framework created by deepset for building production-ready search systems powered by LLMs.", "qas": [{"id": "1", "question": "Who created Haystack?", "answers": [{"text": "deepset", "answer_start": 43}], "is_impossible": false}]}]}]}

In [34]:
# Placing it in a directory called 'data'

# Creating a folder and moving the dataset into it
# import os
# os.makedirs("data", exist_ok=True)
# os.rename("small_squad_dataset.json", "data/small_squad_dataset.json")

# Haystack expects training data to be inside a folder.
# Now everything is ready for training

In [40]:
!mkdir -p data/converted
!wget -O data/converted/train.txt https://raw.githubusercontent.com/sarabou/haystack-data/main/train.txt

--2025-04-05 11:26:23--  https://raw.githubusercontent.com/sarabou/haystack-data/main/train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-04-05 11:26:23 ERROR 404: Not Found.



In [42]:
# Deleting any broken or empty file (just in case)

!rm -f data/converted/train.txt

In [43]:
!mkdir -p data/converted
!curl -L https://huggingface.co/datasets/rajpurkar/squad/resolve/main/train-v1.1.json -o data/train-v1.1.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    15  100    15    0     0     57      0 --:--:-- --:--:-- --:--:--    57


In [44]:
# 5. Training the Reader on Your Dataset

# # Training the model on the small dataset

# reader.train(
#     data_dir="data/converted",                          # Folder containing the dataset
#     train_filename="train.txt",               # Dataset file
#     use_gpu=True,                             # Use GPU to speed up training
#     n_epochs=1,                               # Just one epoch for now
#     save_dir="my_model"                       # Folder to save the fine-tuned model
# )

# n_epochs=1 keeps training quick — you can increase this later for better performance.
# The model will be saved in the "my_model" folder after training.

In [46]:
# instead of using FARMReader.train(), we’ll skip fine-tuning and go straight to using the model since Haystack's fine-tuning pipeline is broken in this version
#  (due to deprecated converters).

# Realistic workaround: Use FARMReader pre-trained model directly to answer questions (no fine-tune)

from haystack.pipelines import ExtractiveQAPipeline
from haystack.document_stores import InMemoryDocumentStore
from haystack.nodes import FARMReader, BM25Retriever

# Creating a document store with BM25 indexing enabled
doc_store = InMemoryDocumentStore(use_bm25=True)

# Writing a simple document into the store
docs = [{"content": "Haystack is an open-source framework by deepset for building search systems."}]
doc_store.write_documents(docs)

# Initializing the BM25 retriever
retriever = BM25Retriever(document_store=doc_store)

# Initializing the FARMReader
reader = FARMReader(model_name_or_path="deepset/roberta-base-squad2", use_gpu=True)

# Building the extractive QA pipeline
pipe = ExtractiveQAPipeline(reader, retriever)

# Asking a question
prediction = pipe.run(
    query="Who created Haystack?",
    params={"Retriever": {"top_k": 1}, "Reader": {"top_k": 1}}
)

# Displaying the top answer
print("Answer:", prediction["answers"][0].answer)


Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.85s/ Batches]

Answer: deepset
